# Bayes' theorem & the base-rate fallacy — interactive grid + death-cause chart

Each of the $N^2$ squares is one person, coloured by **vaccination status × outcome**:

| colour | meaning | variable |
|---|---|---|
| 🟧 orange | vaccinated · healthy | `vaccine_healthy` |
| ⬜ grey | not vaccinated · healthy | `not_vaccine_healthy` |
| 🟥 red | vaccinated · died (side effects) | `vaccine_die` |
| ⬛ black | not vaccinated · died (disease) | `not_vaccine_die` |

The grid fills **row by row** in this order: vaccinated dead → vaccinated healthy → not-vaccinated dead → not-vaccinated healthy.

### Three sliders
- **`immunity`** — fraction of the population vaccinated, $P(\text{vacc})$. Range 0–1.
- **`vaccine_safety`** ($s$) — the higher, the fewer side-effect deaths. It sets $P(\text{die}\mid\text{vacc}) = 10^{-s}$ (log scale). Default $s=5 \Rightarrow 10^{-5}$.
- **`disease_lethality`** — the higher, the deadlier the disease. It *is* $P(\text{die}\mid\text{not vacc})$, linear 0–1. Default 0.10.

### Second view — death-cause bar chart
Two bars matching the grid's death colours: **red** = side-effect deaths (vaccinated), **black** = disease deaths (not vaccinated). Y axis is the **frequency** (number of deaths); each bar is labelled with its exact count. Both views share one cell and redraw together.

---
**Crossover to hunt for:** with the defaults, $P(\text{vaccinated}\mid\text{died})$ passes 0.5 at `immunity ≈ 0.9999` — most dead are vaccinated, yet per-person vaccine risk stays ~10,000× lower than disease risk. That is the base-rate fallacy.

In [61]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches
import ipywidgets as widgets
from ipywidgets import FloatSlider
from IPython.display import display

%matplotlib inline

In [62]:
# ---- Defaults ------------------------------------------------------------
N                 = 1000    # grid side -> N**2 = 1,000,000 people
immunity_default  = 0.999    # fraction vaccinated, P(vacc)
safety_default    = 5.0     # P(die | vacc) = 10**(-safety)  -> 1e-5
lethality_default = 0.01    # P(die | not vacc), linear

TOTAL = N * N

In [63]:
def compute_population(immunity, prior_disease, prior_side_effects, N=N):
    """Integer head-counts for the four categories (sum exactly to N**2)."""
    total = N * N
    n_vacc = int(round(immunity * total))
    n_not  = total - n_vacc
    vaccine_die         = int(round(n_vacc * prior_side_effects))
    vaccine_healthy     = n_vacc - vaccine_die
    not_vaccine_die     = int(round(n_not * prior_disease))
    not_vaccine_healthy = n_not - not_vaccine_die
    return dict(n_vacc=n_vacc, n_not=n_not,
                vaccine_healthy=vaccine_healthy, vaccine_die=vaccine_die,
                not_vaccine_healthy=not_vaccine_healthy, not_vaccine_die=not_vaccine_die)


# Fill sequence -> code. reshape() is C-order, so this fills row by row.
FILL_ORDER = [(0, 'vaccine_die'),         # vaccinated dead       -> red
              (1, 'vaccine_healthy'),     # vaccinated healthy    -> orange
              (2, 'not_vaccine_die'),     # not-vaccinated dead   -> black
              (3, 'not_vaccine_healthy')] # not-vaccinated healthy -> grey

def build_grid(c, N=N):
    grid = np.zeros(N * N, dtype=np.uint8); i = 0
    for code, key in FILL_ORDER:
        grid[i:i + c[key]] = code; i += c[key]
    assert i == N * N
    return grid.reshape(N, N)

In [ ]:
cmap = ListedColormap(['#d62728', '#ff7f0e', '#000000', '#9e9e9e'])  # red, orange, black, grey
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], 4)

grid_legend = [
    mpatches.Patch(color='#ff7f0e', label='vaccinated · healthy'),
    mpatches.Patch(color='#9e9e9e', label='not vaccinated · healthy'),
    mpatches.Patch(color='#d62728', label='vaccinated · died (side effects)'),
    mpatches.Patch(color='#000000', label='not vaccinated · died (disease)'),
]

# --- sliders ---
imm_slider = FloatSlider(value=immunity_default, min=0.0, max=1.0, step=0.001,
                         description='immunity', readout_format='.3f',
                         continuous_update=True, layout=widgets.Layout(width='60%'))
saf_slider = FloatSlider(value=safety_default, min=2.0, max=7.0, step=0.1,
                         description='vaccine safety', readout_format='.1f',
                         continuous_update=True, layout=widgets.Layout(width='60%'))
let_slider = FloatSlider(value=lethality_default, min=0.0, max=1.0, step=0.01,
                         description='disease lethality', readout_format='.2f',
                         continuous_update=True, layout=widgets.Layout(width='60%'))

main_out = widgets.Output()

# ONE persistent figure, reused on every update so plots never pile up.
fig_main, (ax_grid, ax_bar) = plt.subplots(1, 2, figsize=(13.5, 6.6),
                                           gridspec_kw={'width_ratios': [1.3, 1]})
plt.close(fig_main)   # detach from pyplot; we display it manually into main_out


def render_main(change=None):
    immunity, vaccine_safety, disease_lethality = (imm_slider.value,
                                                   saf_slider.value,
                                                   let_slider.value)
    prior_side_effects = 10 ** (-vaccine_safety)
    prior_disease      = disease_lethality
    c = compute_population(immunity, prior_disease, prior_side_effects)
    grid = build_grid(c)

    total_die   = c['vaccine_die'] + c['not_vaccine_die']
    p_vacc_dead = c['vaccine_die'] / total_die if total_die else 0.0

    ax_grid.clear(); ax_bar.clear()

    # grid
    ax_grid.imshow(grid, cmap=cmap, norm=norm, interpolation='nearest', aspect='equal')
    ax_grid.set_xticks([]); ax_grid.set_yticks([])
    ax_grid.set_title('Population', fontsize=12)
    ax_grid.legend(handles=grid_legend, loc='upper center',
                   bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, fontsize=8.5)

    # death-cause bar chart
    labels = ['side effects\n(vaccinated)', 'disease\n(not vaccinated)']
    values = [c['vaccine_die'], c['not_vaccine_die']]
    bars = ax_bar.bar(labels, values, color=['#d62728', '#000000'], width=0.6)
    top = max(values) if max(values) > 0 else 1
    for b, v in zip(bars, values):
        ax_bar.text(b.get_x() + b.get_width() / 2, b.get_height() + top * 0.01,
                    f"{v:,}", ha='center', va='bottom', fontsize=10)
    ax_bar.set_ylabel('frequency (number of deaths)')
    ax_bar.set_title('Deaths by cause', fontsize=12)
    ax_bar.set_ylim(0, top * 1.15)
    ax_bar.spines[['top', 'right']].set_visible(False)

    fig_main.suptitle(
        f"immunity = {immunity:.4f}   |   "
        f"P(die | vaccinated) = {prior_side_effects:.1e}   "
        f"P(die | not vaccinated) = {prior_disease:.2f}\n"
        f"vaccinated: {c['n_vacc']:,}   not vaccinated: {c['n_not']:,}   "
        f"total deaths: {total_die:,}   |   "
        f"P(vaccinated | died) = {p_vacc_dead:.3f}",
        fontsize=11)
    fig_main.tight_layout(rect=[0, 0, 1, 0.93])

    with main_out:
        main_out.clear_output(wait=True)   # replace previous frame, no stacking
        display(fig_main)


for _s in (imm_slider, saf_slider, let_slider):
    _s.observe(render_main, names='value')

render_main()                                              # initial draw
display(widgets.VBox([imm_slider, saf_slider, let_slider, main_out]))

### How the sliders interact

- **`vaccine_safety` down** ($s=3 \Rightarrow 10^{-3}$): the red bar shoots up — at `immunity = 0.99` side-effect deaths jump from ~10 to ~990, rivalling disease deaths.
- **`disease_lethality` up**: the black bar grows; vaccination looks ever more protective.
- **`immunity` up toward 1**: the unvaccinated pool shrinks, disease deaths fall toward zero, and eventually *every* death is a vaccinated person.

$P(\text{die}\mid\text{vaccinated})$ tracks `vaccine_safety` alone, while $P(\text{vaccinated}\mid\text{died})$ is dragged around mostly by `immunity`. Bayes links them:

$$P(\text{vacc}\mid\text{died}) = \frac{P(\text{died}\mid\text{vacc})\,P(\text{vacc})}{P(\text{died})}.$$

---
## Optimizing the immunity level

Fix the vaccine's safety and the disease's lethality; then ask: **which immunity level minimizes total deaths?**

$$\text{deaths}(x) = \underbrace{x\,N^2\,p_{se}}_{\text{side effects}} \;+\; \underbrace{(1-x)\,N^2\,p_d}_{\text{disease}}, \qquad x=\text{immunity},\ p_{se}=10^{-s},\ p_d=\text{lethality}.$$

Both terms are **linear** in $x$, so total deaths is a straight line and its minimum lies at an **endpoint**: vaccinate everyone ($x=1$) when the vaccine is safer than the disease ($p_{se} < p_d$), or no one ($x=0$) otherwise. The decision flips exactly at $10^{-s} = \text{lethality}$. The code below finds the minimum by a numerical scan (so it would still work for a non-linear model) and reports it. Use the sliders to fix safety and lethality and watch the optimum flip.

In [ ]:
opt_saf_slider = FloatSlider(value=safety_default, min=2.0, max=7.0, step=0.1,
                             description='vaccine safety', readout_format='.1f',
                             continuous_update=True, layout=widgets.Layout(width='60%'))
opt_let_slider = FloatSlider(value=lethality_default, min=0.0, max=1.0, step=0.01,
                             description='disease lethality', readout_format='.2f',
                             continuous_update=True, layout=widgets.Layout(width='60%'))

opt_out = widgets.Output()
fig_opt, ax_opt = plt.subplots(figsize=(9.5, 6.0))
plt.close(fig_opt)


def render_opt(change=None, N=N, points=2001):
    vaccine_safety, disease_lethality = opt_saf_slider.value, opt_let_slider.value
    p_se = 10 ** (-vaccine_safety)
    p_d  = disease_lethality
    total_pop = N * N

    x    = np.linspace(0.0, 1.0, points)        # immunity = optimization variable
    side = x * total_pop * p_se                 # side-effect deaths (vaccinated)
    dis  = (1.0 - x) * total_pop * p_d          # disease deaths (not vaccinated)
    tot  = side + dis                           # total deaths -> minimise

    k = int(np.argmin(tot))
    x_opt, min_deaths = x[k], tot[k]
    flat    = (tot.max() - tot.min()) < 1e-6 * max(tot.mean(), 1.0)
    x_cross = p_d / (p_se + p_d) if (p_se + p_d) > 0 else float('nan')

    ax_opt.clear()
    ax_opt.plot(x, side, color='#d62728', lw=2,   label='side-effect deaths (vaccinated)')
    ax_opt.plot(x, dis,  color='#000000', lw=2,   label='disease deaths (not vaccinated)')
    ax_opt.plot(x, tot,  color='#1f77b4', lw=2.6, label='total deaths')
    if not flat:
        ax_opt.axvline(x_opt, color='grey', ls='--', lw=1)
        ax_opt.scatter([x_opt], [min_deaths], color='#1f77b4', s=70,
                       zorder=5, edgecolor='white')
        ax_opt.annotate(f"min = {min_deaths:,.0f}\n@ immunity = {x_opt:.3f}",
                        xy=(x_opt, min_deaths), xytext=(0.45, 0.55),
                        textcoords='axes fraction', fontsize=10,
                        arrowprops=dict(arrowstyle='->', color='grey'))
    ax_opt.set_xlabel('immunity level  (fraction vaccinated)')
    ax_opt.set_ylabel('number of deaths')
    ax_opt.set_xlim(0, 1)
    ax_opt.set_title(f"Deaths vs immunity   |   P(die|vacc) = {p_se:.1e},   "
                     f"P(die|not vacc) = {p_d:.2f}", fontsize=11)
    ax_opt.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
    ax_opt.spines[['top', 'right']].set_visible(False)
    fig_opt.tight_layout()

    if flat:
        msg = (f"Vaccine and disease are equally deadly (p = {p_se:.2e}); "
               f"total deaths = {tot[0]:,.0f} for EVERY immunity level "
               f"-> the optimum is indifferent.")
    else:
        decision = "vaccinate everyone" if x_opt > 0.5 else "vaccinate no one"
        msg = (f"Minimum total deaths = {min_deaths:,.0f}, reached at "
               f"immunity = {x_opt:.3f}  ->  {decision}. "
               f"(The two causes are equal at immunity = {x_cross:.4f}.)")

    with opt_out:
        opt_out.clear_output(wait=True)
        display(fig_opt)
        print(msg)


for _s in (opt_saf_slider, opt_let_slider):
    _s.observe(render_opt, names='value')

render_opt()
display(widgets.VBox([opt_saf_slider, opt_let_slider, opt_out]))